# Coconut: Chain of Continuous Thought

This notebook implements the **Coconut (Chain of Continuous Thought)** paradigm from the paper:

> **Training Large Language Models to Reason in a Continuous Latent Space**  
> Shibo Hao et al., Meta AI (2024)  
> [arXiv:2412.06769](https://arxiv.org/abs/2412.06769)

## Key Contributions

1. **Latent Reasoning**: Instead of generating explicit chain-of-thought tokens, the model reasons in a continuous latent space by feeding hidden states back as inputs.

2. **Continuous Thoughts**: The last hidden state is used as a "continuous thought" representation, which is directly fed back to the model as the next input embedding.

3. **BFS-like Reasoning**: Continuous thoughts can encode multiple alternative next steps simultaneously, enabling breadth-first search over reasoning paths.

4. **Multi-stage Curriculum Training**: Gradually transitions from explicit CoT to latent reasoning through curriculum learning.

## Architecture Overview

```
Language Mode:    [Token] -> [Embedding] -> [Transformer] -> [Hidden] -> [Token]
                                                                  ↓
Latent Mode:      [<bot>] -> [Continuous Thought] ───────────────────────→ [<eot>]
                               ↑                                            |
                               └────────── [Hidden State] ←─────────────────┘
```

## Setup and Installation

In [ ]:
%%capture
import os

# Install dependencies
if "COLAB_" in "".join(os.environ.keys()):
    !pip install transformers accelerate bitsandbytes datasets torch
else:
    !pip install transformers accelerate datasets torch --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from transformers.modeling_outputs import CausalLMOutputWithPast
from typing import Optional, Tuple, List, Dict, Any
import json
from dataclasses import dataclass
import copy

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## Configuration

In [ ]:
@dataclass
class CoconutConfig:
    """Configuration for Coconut training and inference."""
    
    # Model settings
    model_name: str = "meta-llama/Llama-3.2-1B"  # Base model to use
    
    # Special tokens for latent mode
    bot_token: str = "<bot>"  # Beginning of thought
    eot_token: str = "<eot>"  # End of thought
    
    # Training settings
    c_thought: int = 1  # Number of continuous thought vectors per reasoning step
    epochs_per_stage: int = 3  # Training epochs per curriculum stage
    max_latent_stage: int = 5  # Maximum number of latent stages
    
    # Optimization
    learning_rate: float = 1e-5
    weight_decay: float = 0.01
    batch_size: int = 4
    gradient_accumulation_steps: int = 4
    max_seq_length: int = 512
    
    # Curriculum settings
    pad_latent_to_max: bool = True  # Pad continuous thoughts when steps < stage index
    uniform_prob: float = 0.0  # Probability of mixing data from other stages
    
    # Device
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

config = CoconutConfig()
print(f"Using device: {config.device}")

## Coconut Model Implementation

The core innovation of Coconut is replacing discrete token transitions with continuous hidden state transitions during reasoning.

### Key Mechanism:
1. **Language Mode**: Standard token → embedding → transformer → output token
2. **Latent Mode**: Between `<bot>` and `<eot>`, the last hidden state is fed back as the next input embedding

In [ ]:
class CoconutModel(nn.Module):
    """
    Coconut: Chain of Continuous Thought Model
    
    This model wraps a standard causal LM and adds the ability to reason
    in a continuous latent space by feeding hidden states back as inputs.
    """
    
    def __init__(
        self,
        base_model: AutoModelForCausalLM,
        tokenizer: AutoTokenizer,
        config: CoconutConfig
    ):
        super().__init__()
        self.model = base_model
        self.tokenizer = tokenizer
        self.config = config
        
        # Get model dimensions
        self.hidden_size = base_model.config.hidden_size
        
        # Get special token IDs
        self.bot_token_id = tokenizer.convert_tokens_to_ids(config.bot_token)
        self.eot_token_id = tokenizer.convert_tokens_to_ids(config.eot_token)
        
        print(f"Initialized CoconutModel")
        print(f"  Hidden size: {self.hidden_size}")
        print(f"  <bot> token ID: {self.bot_token_id}")
        print(f"  <eot> token ID: {self.eot_token_id}")
    
    def get_input_embeddings(self):
        """Get the input embedding layer."""
        return self.model.get_input_embeddings()
    
    def find_latent_positions(self, input_ids: torch.Tensor) -> List[List[Tuple[int, int]]]:
        """
        Find positions between <bot> and <eot> tokens where latent reasoning occurs.
        
        Returns:
            List of (start, end) tuples for each batch element indicating
            the positions that should use continuous thought.
        """
        batch_size = input_ids.size(0)
        positions = []
        
        for b in range(batch_size):
            seq = input_ids[b].tolist()
            batch_positions = []
            
            # Find all <bot>...<eot> pairs
            i = 0
            while i < len(seq):
                if seq[i] == self.bot_token_id:
                    start = i + 1  # Position after <bot>
                    # Find matching <eot>
                    j = i + 1
                    while j < len(seq) and seq[j] != self.eot_token_id:
                        j += 1
                    if j < len(seq):
                        end = j  # Position of <eot>
                        batch_positions.append((start, end))
                    i = j + 1
                else:
                    i += 1
            
            positions.append(batch_positions)
        
        return positions
    
    def replace_with_continuous_thoughts(
        self,
        inputs_embeds: torch.Tensor,
        hidden_states: torch.Tensor,
        latent_positions: List[List[Tuple[int, int]]]
    ) -> torch.Tensor:
        """
        Replace token embeddings at latent positions with continuous thoughts
        (hidden states from previous positions).
        
        Args:
            inputs_embeds: Token embeddings (batch, seq_len, hidden_size)
            hidden_states: Last layer hidden states (batch, seq_len, hidden_size)
            latent_positions: List of (start, end) tuples for each batch
            
        Returns:
            Modified input embeddings with continuous thoughts inserted
        """
        new_embeds = inputs_embeds.clone()
        
        for b, positions in enumerate(latent_positions):
            for start, end in positions:
                # For each position in the latent range, use the hidden state
                # from the previous position as the input embedding
                for pos in range(start, end):
                    if pos > 0:
                        # Use hidden state from previous position as continuous thought
                        new_embeds[b, pos] = hidden_states[b, pos - 1]
        
        return new_embeds
    
    def forward(
        self,
        input_ids: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        inputs_embeds: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        use_latent_mode: bool = True,
        num_latent_iterations: int = 1,
        **kwargs
    ) -> CausalLMOutputWithPast:
        """
        Forward pass with optional latent reasoning.
        
        When use_latent_mode is True, positions between <bot> and <eot> tokens
        will use continuous thought (hidden state feedback) instead of
        discrete token embeddings.
        """
        if input_ids is not None and inputs_embeds is None:
            inputs_embeds = self.get_input_embeddings()(input_ids)
        
        if not use_latent_mode:
            # Standard forward pass without latent reasoning
            return self.model(
                inputs_embeds=inputs_embeds,
                attention_mask=attention_mask,
                labels=labels,
                output_hidden_states=True,
                **kwargs
            )
        
        # Find positions where latent reasoning should occur
        if input_ids is not None:
            latent_positions = self.find_latent_positions(input_ids)
        else:
            latent_positions = [[] for _ in range(inputs_embeds.size(0))]
        
        # Check if any latent positions exist
        has_latent = any(len(pos) > 0 for pos in latent_positions)
        
        if not has_latent:
            # No latent positions, standard forward
            return self.model(
                inputs_embeds=inputs_embeds,
                attention_mask=attention_mask,
                labels=labels,
                output_hidden_states=True,
                **kwargs
            )
        
        # Iterative forward with continuous thought feedback
        current_embeds = inputs_embeds
        all_hidden_states = None
        
        for iteration in range(num_latent_iterations):
            # Forward pass
            outputs = self.model(
                inputs_embeds=current_embeds,
                attention_mask=attention_mask,
                output_hidden_states=True,
                **kwargs
            )
            
            # Get last hidden states
            hidden_states = outputs.hidden_states[-1]
            all_hidden_states = hidden_states
            
            # Replace embeddings at latent positions with hidden states
            if iteration < num_latent_iterations - 1:
                current_embeds = self.replace_with_continuous_thoughts(
                    inputs_embeds,
                    hidden_states,
                    latent_positions
                )
        
        # Final forward pass to get logits with labels
        final_outputs = self.model(
            inputs_embeds=current_embeds,
            attention_mask=attention_mask,
            labels=labels,
            output_hidden_states=True,
            **kwargs
        )
        
        return final_outputs
    
    @torch.no_grad()
    def generate_with_latent_thought(
        self,
        input_ids: torch.Tensor,
        num_thoughts: int = 3,
        max_new_tokens: int = 100,
        temperature: float = 1.0,
        **kwargs
    ) -> torch.Tensor:
        """
        Generate text with latent reasoning.
        
        The model first performs `num_thoughts` iterations of latent reasoning
        (continuous thought), then generates output tokens.
        """
        device = input_ids.device
        batch_size = input_ids.size(0)
        
        # Get initial embeddings
        inputs_embeds = self.get_input_embeddings()(input_ids)
        attention_mask = torch.ones_like(input_ids)
        
        # Add <bot> token
        bot_embed = self.get_input_embeddings()(
            torch.tensor([[self.bot_token_id]], device=device).expand(batch_size, -1)
        )
        inputs_embeds = torch.cat([inputs_embeds, bot_embed], dim=1)
        attention_mask = torch.cat([
            attention_mask,
            torch.ones(batch_size, 1, device=device, dtype=attention_mask.dtype)
        ], dim=1)
        
        # Perform latent reasoning iterations
        for thought_idx in range(num_thoughts):
            outputs = self.model(
                inputs_embeds=inputs_embeds,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
            
            # Get the last hidden state as continuous thought
            last_hidden = outputs.hidden_states[-1][:, -1:, :]
            
            # Append continuous thought to embeddings
            inputs_embeds = torch.cat([inputs_embeds, last_hidden], dim=1)
            attention_mask = torch.cat([
                attention_mask,
                torch.ones(batch_size, 1, device=device, dtype=attention_mask.dtype)
            ], dim=1)
        
        # Add <eot> token
        eot_embed = self.get_input_embeddings()(
            torch.tensor([[self.eot_token_id]], device=device).expand(batch_size, -1)
        )
        inputs_embeds = torch.cat([inputs_embeds, eot_embed], dim=1)
        attention_mask = torch.cat([
            attention_mask,
            torch.ones(batch_size, 1, device=device, dtype=attention_mask.dtype)
        ], dim=1)
        
        # Generate output tokens autoregressively
        generated_ids = []
        
        for _ in range(max_new_tokens):
            outputs = self.model(
                inputs_embeds=inputs_embeds,
                attention_mask=attention_mask
            )
            
            # Get logits for the last position
            logits = outputs.logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            generated_ids.append(next_token)
            
            # Check for EOS
            if (next_token == self.tokenizer.eos_token_id).all():
                break
            
            # Add new token embedding
            next_embed = self.get_input_embeddings()(next_token)
            inputs_embeds = torch.cat([inputs_embeds, next_embed], dim=1)
            attention_mask = torch.cat([
                attention_mask,
                torch.ones(batch_size, 1, device=device, dtype=attention_mask.dtype)
            ], dim=1)
        
        # Combine input and generated tokens
        generated_ids = torch.cat(generated_ids, dim=1)
        full_ids = torch.cat([input_ids, generated_ids], dim=1)
        
        return full_ids

## Data Processing for Multi-Stage Curriculum

Coconut uses a multi-stage curriculum where reasoning steps gradually transition from explicit tokens to continuous latent representations.

### Data Format:
```json
{
    "question": "What is 2 + 3?",
    "answer": "5",
    "steps": ["First, I add 2 and 3", "2 + 3 = 5", "The answer is 5"]
}
```

### Curriculum Stages:
- **Stage 0**: Full CoT with all explicit reasoning steps
- **Stage k**: First k reasoning steps replaced with k continuous thought positions

In [ ]:
class CoconutDataProcessor:
    """
    Process data for multi-stage Coconut curriculum training.
    """
    
    def __init__(
        self,
        tokenizer: AutoTokenizer,
        config: CoconutConfig
    ):
        self.tokenizer = tokenizer
        self.config = config
    
    def format_for_stage(
        self,
        question: str,
        answer: str,
        steps: List[str],
        stage: int
    ) -> Dict[str, Any]:
        """
        Format a data point for a specific curriculum stage.
        
        Args:
            question: The input question
            answer: The final answer
            steps: List of reasoning steps
            stage: Current curriculum stage (0 = full CoT, k = k steps replaced)
            
        Returns:
            Dictionary with 'input_text', 'target_text', and 'num_latent_positions'
        """
        num_steps = len(steps)
        num_latent = min(stage, num_steps)  # Number of steps to replace with latent
        
        if stage == 0:
            # Full CoT: Question -> [All Steps] -> Answer
            cot_text = " ".join(steps)
            input_text = f"Question: {question}\nLet me think step by step.\n{cot_text}\nAnswer: {answer}"
            target_text = input_text
            num_latent_positions = 0
        else:
            # Replace first `num_latent` steps with latent positions
            remaining_steps = steps[num_latent:]
            remaining_cot = " ".join(remaining_steps) if remaining_steps else ""
            
            # Latent positions: <bot> [c_thought positions per step] <eot>
            num_latent_positions = num_latent * self.config.c_thought
            
            # Pad if needed
            if self.config.pad_latent_to_max and num_latent < stage:
                num_latent_positions = stage * self.config.c_thought
            
            # Create placeholder tokens for latent positions (will be replaced during training)
            latent_placeholders = " ".join(["[THOUGHT]" for _ in range(num_latent_positions)])
            
            input_text = f"Question: {question}\n{self.config.bot_token} {latent_placeholders} {self.config.eot_token}"
            if remaining_cot:
                input_text += f"\n{remaining_cot}"
            input_text += f"\nAnswer: {answer}"
            target_text = input_text
        
        return {
            "input_text": input_text,
            "target_text": target_text,
            "num_latent_positions": num_latent_positions,
            "stage": stage
        }
    
    def prepare_batch(
        self,
        data_points: List[Dict],
        stage: int
    ) -> Dict[str, torch.Tensor]:
        """
        Prepare a batch of data for training at a specific stage.
        """
        formatted = [
            self.format_for_stage(
                d["question"],
                d["answer"],
                d["steps"],
                stage
            )
            for d in data_points
        ]
        
        texts = [f["input_text"] for f in formatted]
        
        encodings = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.config.max_seq_length,
            return_tensors="pt"
        )
        
        # Labels for causal LM (same as input_ids, with -100 for padding)
        labels = encodings["input_ids"].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            "input_ids": encodings["input_ids"],
            "attention_mask": encodings["attention_mask"],
            "labels": labels
        }

## Example Dataset: Logical Reasoning

Let's create a simple dataset for demonstration.

In [ ]:
# Example dataset for multi-hop logical reasoning (similar to ProntoQA)
EXAMPLE_DATA = [
    {
        "question": "Rex is a lion. Every lion is a mammal. Every mammal is an animal. Is Rex an animal?",
        "answer": "Yes",
        "steps": [
            "Rex is a lion.",
            "Every lion is a mammal, so Rex is a mammal.",
            "Every mammal is an animal, so Rex is an animal."
        ]
    },
    {
        "question": "Tweety is a bird. Every bird can fly. Every thing that can fly has wings. Does Tweety have wings?",
        "answer": "Yes",
        "steps": [
            "Tweety is a bird.",
            "Every bird can fly, so Tweety can fly.",
            "Every thing that can fly has wings, so Tweety has wings."
        ]
    },
    {
        "question": "Fido is a dog. Every dog is loyal. Is Fido loyal?",
        "answer": "Yes",
        "steps": [
            "Fido is a dog.",
            "Every dog is loyal, so Fido is loyal."
        ]
    },
    {
        "question": "Bob is a student. Every student studies. Every person who studies learns. Does Bob learn?",
        "answer": "Yes",
        "steps": [
            "Bob is a student.",
            "Every student studies, so Bob studies.",
            "Every person who studies learns, so Bob learns."
        ]
    },
    {
        "question": "Rain causes wet ground. It is raining. Is the ground wet?",
        "answer": "Yes",
        "steps": [
            "It is raining.",
            "Rain causes wet ground, so the ground is wet."
        ]
    }
]

print(f"Created {len(EXAMPLE_DATA)} example data points")
print(f"\nExample data point:")
print(json.dumps(EXAMPLE_DATA[0], indent=2))

## Model Loading and Setup

In [ ]:
def setup_model_and_tokenizer(config: CoconutConfig):
    """
    Load base model and tokenizer, add special tokens for Coconut.
    """
    print(f"Loading model: {config.model_name}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        config.model_name,
        trust_remote_code=True
    )
    
    # Add special tokens for latent mode
    special_tokens = {
        "additional_special_tokens": [config.bot_token, config.eot_token]
    }
    
    # Set pad token if not exists
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    num_added = tokenizer.add_special_tokens(special_tokens)
    print(f"Added {num_added} special tokens")
    
    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float16 if config.device == "cuda" else torch.float32,
        device_map="auto" if config.device == "cuda" else None,
        trust_remote_code=True
    )
    
    # Resize embeddings for new tokens
    model.resize_token_embeddings(len(tokenizer))
    
    # Create Coconut wrapper
    coconut_model = CoconutModel(model, tokenizer, config)
    
    return coconut_model, tokenizer

# Load model (uncomment to run)
# coconut_model, tokenizer = setup_model_and_tokenizer(config)

## Multi-Stage Curriculum Training

The training process follows a curriculum:
1. **Stage 0**: Train on full CoT data
2. **Stage 1**: Replace first step with latent, train on remaining steps
3. **Stage k**: Replace first k steps with latent thoughts

At each stage, the optimizer state is reset.

In [ ]:
class CoconutTrainer:
    """
    Multi-stage curriculum trainer for Coconut.
    """
    
    def __init__(
        self,
        model: CoconutModel,
        tokenizer: AutoTokenizer,
        config: CoconutConfig,
        train_data: List[Dict]
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.config = config
        self.train_data = train_data
        self.data_processor = CoconutDataProcessor(tokenizer, config)
    
    def train_stage(self, stage: int) -> float:
        """
        Train for one curriculum stage.
        
        Returns:
            Average loss for the stage
        """
        print(f"\n{'='*50}")
        print(f"Training Stage {stage}")
        print(f"{'='*50}")
        
        # Reset optimizer at each stage (as per paper)
        optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=self.config.learning_rate,
            weight_decay=self.config.weight_decay
        )
        
        self.model.train()
        total_loss = 0
        num_steps = 0
        
        for epoch in range(self.config.epochs_per_stage):
            epoch_loss = 0
            
            # Create batches
            for i in range(0, len(self.train_data), self.config.batch_size):
                batch_data = self.train_data[i:i + self.config.batch_size]
                
                # Prepare batch for current stage
                batch = self.data_processor.prepare_batch(batch_data, stage)
                
                # Move to device
                batch = {k: v.to(self.config.device) for k, v in batch.items()}
                
                # Forward pass
                outputs = self.model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    labels=batch["labels"],
                    use_latent_mode=(stage > 0)
                )
                
                loss = outputs.loss / self.config.gradient_accumulation_steps
                loss.backward()
                
                epoch_loss += loss.item()
                num_steps += 1
                
                # Gradient accumulation
                if num_steps % self.config.gradient_accumulation_steps == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    optimizer.step()
                    optimizer.zero_grad()
            
            avg_epoch_loss = epoch_loss / max(1, len(self.train_data) // self.config.batch_size)
            total_loss += avg_epoch_loss
            print(f"  Epoch {epoch + 1}/{self.config.epochs_per_stage}, Loss: {avg_epoch_loss:.4f}")
        
        return total_loss / self.config.epochs_per_stage
    
    def train(self):
        """
        Run full multi-stage curriculum training.
        """
        print("Starting Coconut Multi-Stage Curriculum Training")
        print(f"Max latent stages: {self.config.max_latent_stage}")
        print(f"Epochs per stage: {self.config.epochs_per_stage}")
        
        stage_losses = []
        
        for stage in range(self.config.max_latent_stage + 1):
            stage_loss = self.train_stage(stage)
            stage_losses.append(stage_loss)
            print(f"Stage {stage} completed, Average Loss: {stage_loss:.4f}")
        
        print("\n" + "="*50)
        print("Training Complete!")
        print("="*50)
        print("\nStage Losses:")
        for i, loss in enumerate(stage_losses):
            print(f"  Stage {i}: {loss:.4f}")
        
        return stage_losses

## Demonstration: Data Formatting at Different Stages

In [ ]:
# Create a simple tokenizer mock for demonstration
class MockTokenizer:
    def __init__(self):
        pass

mock_tokenizer = MockTokenizer()
processor = CoconutDataProcessor(mock_tokenizer, config)

example = EXAMPLE_DATA[0]
print("Original Data:")
print(f"  Question: {example['question']}")
print(f"  Steps: {example['steps']}")
print(f"  Answer: {example['answer']}")

print("\n" + "="*60)

for stage in range(4):
    formatted = processor.format_for_stage(
        example["question"],
        example["answer"],
        example["steps"],
        stage
    )
    print(f"\nStage {stage} (Latent positions: {formatted['num_latent_positions']}):")
    print("-" * 40)
    print(formatted["input_text"])

## Inference with Latent Reasoning

During inference, the model:
1. Takes the question as input
2. Generates `<bot>` token
3. Performs N iterations of latent reasoning (continuous thought)
4. Generates `<eot>` token
5. Outputs the final answer

In [ ]:
@torch.no_grad()
def inference_with_latent_thought(
    model: CoconutModel,
    tokenizer: AutoTokenizer,
    question: str,
    num_thoughts: int = 3,
    max_new_tokens: int = 50
) -> str:
    """
    Perform inference with latent reasoning.
    
    Args:
        model: Trained Coconut model
        tokenizer: Tokenizer
        question: Input question
        num_thoughts: Number of latent thinking iterations
        max_new_tokens: Maximum tokens to generate
        
    Returns:
        Generated answer
    """
    model.eval()
    
    # Prepare input
    input_text = f"Question: {question}\n"
    inputs = tokenizer(input_text, return_tensors="pt")
    inputs = {k: v.to(model.config.device) for k, v in inputs.items()}
    
    # Generate with latent thought
    output_ids = model.generate_with_latent_thought(
        inputs["input_ids"],
        num_thoughts=num_thoughts,
        max_new_tokens=max_new_tokens
    )
    
    # Decode output
    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    return output_text

# Example usage (uncomment when model is loaded):
# result = inference_with_latent_thought(
#     coconut_model,
#     tokenizer,
#     "Rex is a lion. Every lion is a mammal. Is Rex a mammal?",
#     num_thoughts=3
# )
# print(result)

## Visualization: BFS-like Reasoning in Latent Space

One key advantage of Coconut is that continuous thoughts can encode multiple reasoning paths simultaneously, allowing for implicit breadth-first search.

In [ ]:
def visualize_latent_reasoning():
    """
    Conceptual visualization of BFS-like reasoning in latent space.
    """
    print("\n" + "="*60)
    print("BFS-like Reasoning in Latent Space")
    print("="*60)
    
    print("""
    Traditional CoT (DFS-like):
    ┌─────────┐    ┌─────────┐    ┌─────────┐    ┌─────────┐
    │Question │ -> │ Step 1  │ -> │ Step 2  │ -> │ Answer  │
    └─────────┘    └─────────┘    └─────────┘    └─────────┘
         ↓              ↓              ↓              ↓
    "Rex is a    "Rex is a      "Rex is a      "Yes, Rex
     lion..."     mammal"        animal"        is animal"
    
    Problem: Commits to single path early, hard to backtrack
    
    ════════════════════════════════════════════════════════════
    
    Coconut (BFS-like in Latent Space):
    
    ┌─────────┐    ┌─────────────────────────────────────┐    ┌─────────┐
    │Question │ -> │     Continuous Thought Space        │ -> │ Answer  │
    └─────────┘    │  ┌───────────────────────────────┐  │    └─────────┘
                   │  │ Hidden State encodes MULTIPLE │  │
                   │  │ possible reasoning paths:     │  │
                   │  │                               │  │
                   │  │  Path A: Rex→mammal→animal    │  │
                   │  │  Path B: Rex→lion→predator    │  │
                   │  │  Path C: Rex→vertebrate       │  │
                   │  │                               │  │
                   │  │ Paths are evaluated in        │  │
                   │  │ parallel, wrong ones pruned   │  │
                   │  └───────────────────────────────┘  │
                   └─────────────────────────────────────┘
    
    Advantage: Explores multiple paths, naturally prunes incorrect ones
    """)

visualize_latent_reasoning()

## Training Example (Full Pipeline)

Here's how to run the complete training pipeline:

In [ ]:
def run_training_demo():
    """
    Run a complete training demonstration.
    
    Note: This requires GPU and sufficient memory.
    For demonstration, we use a small model and dataset.
    """
    # Update config for demo
    demo_config = CoconutConfig(
        model_name="meta-llama/Llama-3.2-1B",  # Smaller model for demo
        epochs_per_stage=1,  # Fewer epochs for demo
        max_latent_stage=2,  # Fewer stages for demo
        batch_size=2,
        max_seq_length=256
    )
    
    print("Setting up model...")
    model, tokenizer = setup_model_and_tokenizer(demo_config)
    
    print("\nCreating trainer...")
    trainer = CoconutTrainer(
        model=model,
        tokenizer=tokenizer,
        config=demo_config,
        train_data=EXAMPLE_DATA
    )
    
    print("\nStarting training...")
    stage_losses = trainer.train()
    
    print("\nTesting inference...")
    test_question = "Max is a cat. Every cat is a mammal. Is Max a mammal?"
    result = inference_with_latent_thought(
        model,
        tokenizer,
        test_question,
        num_thoughts=2
    )
    print(f"Question: {test_question}")
    print(f"Answer: {result}")
    
    return model, tokenizer, stage_losses

# Uncomment to run full training demo (requires GPU)
# model, tokenizer, losses = run_training_demo()

## Key Takeaways

### Advantages of Coconut:

1. **Fewer Tokens**: Reasoning happens in latent space, reducing the number of generated tokens

2. **BFS-like Search**: Hidden states can encode multiple reasoning paths simultaneously

3. **End-to-End Training**: Fully differentiable, allows gradient-based optimization through reasoning

4. **Better Planning**: Can handle tasks requiring backtracking better than traditional CoT

### Limitations:

1. **Interpretability**: Latent thoughts are not human-readable

2. **Training Complexity**: Requires multi-stage curriculum training

3. **Task-Specific**: Performance gains are most notable on logical reasoning tasks

### When to Use Coconut:

- Tasks requiring multi-hop reasoning
- Problems where backtracking might be necessary  
- Scenarios where token efficiency matters
- Logical reasoning and planning tasks

## References

1. [Training Large Language Models to Reason in a Continuous Latent Space](https://arxiv.org/abs/2412.06769) - Original Paper

2. [Official Implementation](https://github.com/facebookresearch/coconut) - Facebook Research GitHub

3. [OpenCoconut](https://github.com/casper-hansen/OpenCoconut) - Community Implementation (Archived)